In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Dense
from tensorflow.keras.regularizers import l2

In [3]:
def load_df(company_code):

    # Load the Excel file and read Data from the file
    file_path = company_code + '.xlsx'
    sheet_name = company_code           
    data = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')

    # Remove rows with any NaN values
    data = data.dropna()
    # Reset the index of the DataFrame and drop the old index
    data = data.reset_index(drop=True)
    return data

def repeat_quaterly_data(quaterly_data, daily_data):
    quaterly_data.index = pd.to_datetime(quaterly_data.index)
    daily_data.index = pd.to_datetime(daily_data.index)

    # Resample the quaterly data to daily frequency
    quaterly_data.index = quaterly_data.index + pd.DateOffset(days=1)
    # Using Forward Fill to Resample the Quaterly Data
    # daily_quaterly_data = quaterly_data.resample('D').ffill()
    # quaterly_data.index = quaterly_data.index - pd.offsets.MonthEnd(1) - pd.DateOffset(days=1)
    
    aligned_quaterly_data = quaterly_data.reindex(daily_data.index, method='ffill')
    aligned_quaterly_data = aligned_quaterly_data.dropna()
    return aligned_quaterly_data

# Function to create sequence
# Need to define the sequence length, e.g. using 4 quaters to predict the next quater
def create_sequences(features, targets, seq_length):
    xs = []
    ys = []
    for i in range(len(features)-seq_length):
        x = features[i:(i+seq_length)]
        y = targets.iloc[i+seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

def get_historical_data(ticker, start_date, end_date):
    data = yf.download(ticker,start=start_date, end=end_date)
    return data

# Function to fetch Historical Price data and compute returns
def get_historical_returns(ticker, start_date, end_date):
    data = yf.download(ticker,start=start_date, end=end_date)

    # Calculate daily returns
    # data['Returns'] = data['Close'].pct_change()
    # data = data.dropna()

    # Monthly Return
    monthly_data = data['Close'].resample('M').last()
    monthly_returns = monthly_data.pct_change()
    monthly_returns = monthly_returns.dropna()

    # return data['Returns']
    return monthly_returns

In [15]:
# Setup TimeFrame and Target Firm
ticker = 'MSFT'
company_code = 'MSFT-US' 
start_date = '2017-08-30'
end_date = '2023-09-30'

# Process Ratio Data
data = load_df(company_code)
data = data.set_index('Date').T
data.index = pd.to_datetime(data.index, format='%b \'%y')
data.index = data.index + pd.offsets.MonthEnd()
ratio_data = data.apply(pd.to_numeric)

# Select a few columns
pe_column = 'Price/Earnings'  
pb_column = 'Price/Book Value'  
roa_column = 'Return on Assets' 
roe_column = 'Return on Equity ' 
fcf_column = 'Dividend Yield (%)'
ratio_data = data[[pe_column, pb_column, roa_column, roe_column, fcf_column]]

# print(ratio_data)

# Process Return Data
returns_data = get_historical_returns(ticker, start_date, end_date)
# print(returns_data.index) # DateTime Index(Check)

adjusted_ratio_data = repeat_quaterly_data(ratio_data, returns_data)
# print(adjusted_ratio_data)

# Scale the data
# scaler = MinMaxScaler(feature_range=(0,1))
# scaled_data = scaler.fit_transform(adjusted_ratio_data)
# scaled_data_df = pd.DataFrame(scaled_data, index=adjusted_ratio_data.index)

features = pd.concat([adjusted_ratio_data, returns_data],axis=1)
print(features)

# Create Sequence: use previous 4 quarters to predict the next quarter
seq_length = 6
X, y = create_sequences(features, returns_data, seq_length)
# print(X)


[*********************100%%**********************]  1 of 1 completed
            Price/Earnings  Price/Book Value  Return on Assets  \
Date                                                             
2017-09-30       25.336735          6.414744         10.003878   
2017-10-31       58.589041          8.410996          4.822175   
2017-11-30       58.589041          8.410996          4.822175   
2017-12-31       58.589041          8.410996          4.822175   
2018-01-31       50.705556          8.857588          6.040628   
...                    ...               ...               ...   
2023-05-31       35.164132         12.272605         18.630152   
2023-06-30       35.164132         12.272605         18.630152   
2023-07-31       30.577856         10.630672         19.140756   
2023-08-31       30.577856         10.630672         19.140756   
2023-09-30       30.577856         10.630672         19.140756   

            Return on Equity   Dividend Yield (%)     Close  
Date      

In [16]:
print(X)

[[[ 2.53367350e+01  6.41474400e+00  1.00038780e+01  2.88590730e+01
    2.13451500e+00 -3.74480128e-03]
  [ 5.85890410e+01  8.41099600e+00  4.82217500e+00  1.57478820e+01
    1.89385100e+00  1.16659990e-01]
  [ 5.85890410e+01  8.41099600e+00  4.82217500e+00  1.57478820e+01
    1.89385100e+00  1.19018738e-02]
  [ 5.85890410e+01  8.41099600e+00  4.82217500e+00  1.57478820e+01
    1.89385100e+00  1.62766161e-02]
  [ 5.07055560e+01  8.85758800e+00  6.04062800e+00  1.90792650e+01
    1.80782300e+00  1.10708454e-01]
  [ 5.07055560e+01  8.85758800e+00  6.04062800e+00  1.90792650e+01
    1.80782300e+00 -1.30513153e-02]]

 [[ 5.85890410e+01  8.41099600e+00  4.82217500e+00  1.57478820e+01
    1.89385100e+00  1.16659990e-01]
  [ 5.85890410e+01  8.41099600e+00  4.82217500e+00  1.57478820e+01
    1.89385100e+00  1.19018738e-02]
  [ 5.85890410e+01  8.41099600e+00  4.82217500e+00  1.57478820e+01
    1.89385100e+00  1.62766161e-02]
  [ 5.07055560e+01  8.85758800e+00  6.04062800e+00  1.90792650e+01
    

In [17]:
print(y)

[-0.02666098  0.02465213  0.05688623 -0.00232695  0.07575298  0.0589178
  0.01816078 -0.06610129  0.03819869 -0.08404725  0.02815793  0.07277601
  0.05275376  0.10734275 -0.05298626  0.08311777  0.01724393  0.01166798
  0.00848686  0.03121626  0.0558695   0.04174919  0.07945465 -0.04828762
 -0.0265415   0.13632616  0.02254335  0.11055932  0.00737065  0.1000927
 -0.06739679 -0.03736985  0.05729247  0.03900589  0.04289187  0.00181065
  0.01458817  0.06960168 -0.00991355  0.08498879  0.05171654  0.05956267
 -0.06611896  0.17629107 -0.00310596  0.01733268 -0.0753449  -0.03919867
  0.03186181 -0.09986705 -0.02035887 -0.05532059  0.09309662 -0.06863999
 -0.10926686 -0.00330609  0.09912546 -0.06004543  0.03331661  0.00649692
  0.1558816   0.06576491  0.06876913  0.03699867 -0.01356667 -0.02429151
 -0.03664269]


In [10]:
#LSTM Model Set Up

# Model architecture
model = tf.keras.Sequential([
    LSTM(512, return_sequences=True),
    Dropout(0.02),
    LSTM(256, return_sequences=True),
    LSTM(128),
    Dense(1, activation='linear', kernel_regularizer=l2(0.0005))
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
model.compile(optimizer=optimizer, loss='mean_squared_error')

# Compile the model
# model.compile(optimizer='adam', loss='mean_squared_error')

In [11]:
# Split data into training and testing sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Train the model
# model.fit(X_train, y_train, epochs=50, batch_size=12, validation_data=(X_test, y_test))

model.fit(X_train, y_train, epochs=500, batch_size=12, validation_data=(X_test, y_test))


Epoch 1/500
5/5 [==============================] - 4s 195ms/step - loss: 0.1234 - val_loss: 0.0126
Epoch 2/500
5/5 [==============================] - 0s 24ms/step - loss: 0.0095 - val_loss: 0.0059
Epoch 3/500
5/5 [==============================] - 0s 26ms/step - loss: 0.0052 - val_loss: 0.0068
Epoch 4/500
5/5 [==============================] - 0s 24ms/step - loss: 0.0054 - val_loss: 0.0111
Epoch 5/500
5/5 [==============================] - 0s 23ms/step - loss: 0.0059 - val_loss: 0.0068
Epoch 6/500
5/5 [==============================] - 0s 22ms/step - loss: 0.0060 - val_loss: 0.0058
Epoch 7/500
5/5 [==============================] - 0s 21ms/step - loss: 0.0042 - val_loss: 0.0065
Epoch 8/500
5/5 [==============================] - 0s 22ms/step - loss: 0.0042 - val_loss: 0.0054
Epoch 9/500
5/5 [==============================] - 0s 22ms/step - loss: 0.0042 - val_loss: 0.0053
Epoch 10/500
5/5 [==============================] - 0s 23ms/step - loss: 0.0040 - val_loss: 0.0057
Epoch 11/500
5/5 [

In [12]:
# Evaluate the model
loss = model.evaluate(X_test, y_test)

# Make predictions
predictions = model.predict(X_train)
print(predictions)
print(y_train)

# Inverse transform to get actual values
# predictions = scaler.inverse_transform(predictions)

2/2 [==============================] - 1s 6ms/step
[[-0.01988676]
 [ 0.02835021]
 [ 0.06041724]
 [-0.00219858]
 [ 0.07624359]
 [ 0.05623896]
 [ 0.01652857]
 [-0.0580195 ]
 [ 0.04029045]
 [-0.07618296]
 [ 0.02870113]
 [ 0.07275791]
 [ 0.05510145]
 [ 0.10831314]
 [-0.04743775]
 [ 0.08100521]
 [ 0.02819724]
 [ 0.01519335]
 [ 0.01588947]
 [ 0.0360337 ]
 [ 0.05961969]
 [ 0.04371833]
 [ 0.07631354]
 [-0.03973752]
 [-0.02140083]
 [ 0.13207404]
 [ 0.02736661]
 [ 0.10439016]
 [ 0.01101327]
 [ 0.09803168]
 [-0.06180957]
 [-0.03106034]
 [ 0.06161438]
 [ 0.03421612]
 [ 0.04585777]
 [ 0.00520169]
 [ 0.01703244]
 [ 0.07133793]
 [-0.00207435]
 [ 0.07480115]
 [ 0.04791428]
 [ 0.05735524]
 [-0.06426398]
 [ 0.1686453 ]
 [-0.00072672]
 [ 0.01759056]
 [-0.0716207 ]
 [-0.05802787]
 [ 0.03097104]
 [-0.0997182 ]
 [-0.03082045]
 [-0.04705514]
 [ 0.09334042]]
[-0.02666098  0.02465213  0.05688623 -0.00232695  0.07575298  0.0589178
  0.01816078 -0.06610129  0.03819869 -0.08404725  0.02815793  0.07277601
  0.0527